### test generate n 

In [3]:
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer, AutoModelForMaskedLM

In [ ]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

llm = LLM(model=model_name)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

messages = [
 {'role': 'user', 'content': 'Explain Optimal Transport'},
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

print(prompt)

In [ ]:
outputs = llm.generate(
    prompts=[prompt, prompt], sampling_params=SamplingParams(temperature=0.7, max_tokens=512, n=5)
)

In [58]:
candidates = []
for o in outputs:
        candidates.append(
            [item.text for item in o.outputs]
        )

In [ ]:
candidates

### Combine data

In [1]:
from datasets import load_dataset, Dataset, concatenate_datasets
from tqdm.auto import tqdm

/opt/conda/envs/violex/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
lst_data_paths = [
    "../results_synthetic/Magpie-Llama-3.1-Pro-MT_rejection-sampling_0-20000.json",
    "../results_synthetic/Magpie-Reasoning_rejection-sampling_0-20000.json",
    "../results_synthetic/SlimOrca-Dedup_rejection-sampling_0-20000.json",
    "../results_synthetic/slm-code-synthetic-v0.1_rejection-sampling_0-50000.json",
    "../results_synthetic/slm-instruct-synthetic-v0.1_rejection-sampling_0-20000.json",
    "../results_synthetic/slm-instruct-synthetic-v0.2_rejection-sampling_0-20000.json",
]

data = concatenate_datasets(
    [Dataset.from_json(f) for f in lst_data_paths]
)

print(len(data))

In [ ]:
data = data.filter(lambda x: len(x["candidates"]) == 8)

In [ ]:
def remove_duplicate_candidates(example):
    candidates = example["candidates"]
    
    unique_contents = set()
    unique_candidates = []
    for c in candidates:
        assistant_message = c[-1]
        assert assistant_message["role"] == "assistant"
        content = assistant_message["content"].strip()
        if content not in unique_contents:
            unique_contents.add(content)
            unique_candidates.append(c)

    example["candidates"] = unique_candidates
    if len(candidates) != len(unique_candidates):
        print(f"Removed duplicate candidates {len(candidates)} -> {len(unique_candidates)}")
        
    return example

data = data.map(remove_duplicate_candidates)

In [52]:
data = data.filter(lambda x: len(x["candidates"]) >= 5)

Filter: 100%|██████████| 149965/149965 [00:22<00:00, 6780.82 examples/s]


In [53]:
data

Dataset({
    features: ['candidates', 'model', 'sampling_params'],
    num_rows: 148605
})

In [56]:
data.to_json("../results_synthetic/combined.json")

Creating json from Arrow format: 100%|██████████| 149/149 [00:38<00:00,  3.84ba/s]


4036621905

In [2]:
data = Dataset.from_json("../results_synthetic/combined.json")

Generating train split: 148605 examples [00:10, 14467.11 examples/s]


In [3]:
data.push_to_hub("slm-research-vn/RS-candidates__slm-4b-sft-v3-chatml")

Uploading the dataset shards: 100%|██████████| 8/8 [01:24<00:00, 10.59s/it]


CommitInfo(commit_url='https://huggingface.co/datasets/slm-research-vn/RS-candidates__slm-4b-sft-v3-chatml/commit/e7b60a28f1b7cbe0ed28fdc3f1fec73b1fc4560d', commit_message='Upload dataset', commit_description='', oid='e7b60a28f1b7cbe0ed28fdc3f1fec73b1fc4560d', pr_url=None, pr_revision=None, pr_num=None)